# Assignment 9: Model Deployment
## Practical Project: Preprocessing, Hyperparameter Tuning & Model Deployment

**Project:** California Housing Price Prediction

### Objective
Build an end-to-end regression workflow that:
1. Loads the California Housing dataset.
2. Preprocesses numerical features using `ColumnTransformer` and `StandardScaler`.
3. Splits the data into training and testing sets (80/20).
4. Builds a `KNeighborsRegressor` pipeline.
5. Tunes hyperparameters using `GridSearchCV` with 5-fold cross-validation and R² scoring.
6. Evaluates the best model on the held-out test set.
7. Saves and reloads the trained pipeline using Joblib.

The notebook is designed to run locally in **VS Code**. It does not use Google Colab commands.

## 1. Environment setup

Install the required packages once from the VS Code terminal:

```bash
python -m pip install pandas numpy scikit-learn joblib jupyter ipykernel
```

For the Streamlit application, also install:

```bash
python -m pip install streamlit
```

Select the Python interpreter belonging to your project virtual environment in VS Code. Then open this notebook and choose that same kernel.

**Expected result:** The packages install successfully. The installation commands are run in the terminal, not in a notebook cell.

## 2. Import libraries

We import:
- `pandas` for creating and inspecting the dataset.
- `joblib` for saving and loading the trained model.
- Scikit-learn tools for data loading, splitting, preprocessing, pipeline construction, tuning, and evaluation.

We also use `Path` to build file paths that work across operating systems.

In [2]:
from pathlib import Path

import pandas as pd
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

## 3. Define project paths

The notebook is stored in the `notebooks` folder. `PROJECT_DIR` points to the main project folder, and the trained model will be saved in its `models` folder.

Creating the folder with `mkdir(..., exist_ok=True)` means Python will create it if it does not already exist and will not raise an error if it is already there.

In [3]:
PROJECT_DIR = Path.cwd()

# If the notebook is running from the notebooks folder, use its parent as the project folder.
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "california_housing_model.pkl"

print("Project directory:", PROJECT_DIR)
print("Model directory:", MODEL_DIR)
print("Model will be saved to:", MODEL_PATH)

Project directory: c:\Users\Admin\Documents\California-housing-price-prediction\california-housing-price-prediction
Model directory: c:\Users\Admin\Documents\California-housing-price-prediction\california-housing-price-prediction\models
Model will be saved to: c:\Users\Admin\Documents\California-housing-price-prediction\california-housing-price-prediction\models\california_housing_model.pkl


## 4. Load the California Housing dataset

`fetch_california_housing()` loads the California Housing regression dataset provided by scikit-learn.

The feature matrix is stored in `housing.data`, and the feature names are stored in `housing.feature_names`. The target values are stored in `housing.target`.

The target is median house value measured in **units of $100,000**, not individual dollars.

In [4]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["MedHouseVal"] = housing.target

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


**How to interpret the output**

- The dataset contains **20,640 rows** and **9 columns**: eight input features and one target.
- `df.head()` displays the first five observations.
- `MedHouseVal` is the value the model will learn to predict.

## 5. Inspect the dataset

Before training, inspect the dataset's dimensions, data types, non-null counts, and descriptive statistics.

- `shape` reports the number of rows and columns.
- `info()` shows column data types and non-null counts.
- `describe()` reports statistics such as mean, standard deviation, minimum, and maximum.

These checks help us understand the data before modelling.

In [5]:
print("Dataset shape:", df.shape)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
df.describe()

Dataset shape: (20640, 9)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

Statistical summary:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


## 6. Separate input features and target

`X` contains the eight predictor variables. `y` contains the target variable, `MedHouseVal`.

We identify numerical columns so that the preprocessing transformer can apply scaling to them.

In [6]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Input features:")
print(numerical_features)

print("\nTarget variable:")
print(y.name)

print("\nNumber of input features:", len(numerical_features))

Input features:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Target variable:
MedHouseVal

Number of input features: 8


**Expected output**

The input features should be `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, and `Longitude`. The target variable should be `MedHouseVal`.

## 7. Create the preprocessing transformer

KNN uses distances to identify nearby observations. Features measured on very different scales can dominate those distance calculations.

`StandardScaler` standardizes each numerical feature using the mean and standard deviation learned from the training data.

`ColumnTransformer` applies the scaler to the selected numerical columns. We place this transformer inside the machine learning pipeline so that it is fitted separately within each cross-validation training fold. This helps prevent data leakage.

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features)
    ]
)

print("Preprocessing transformer created successfully.")

Preprocessing transformer created successfully.


## 8. Split the dataset into training and testing sets

We reserve 20% of the data for final testing and use 80% for training and cross-validation.

- `X_train` and `y_train` are used for model fitting and hyperparameter tuning.
- `X_test` and `y_test` are held aside until the final evaluation.
- `random_state=42` makes the split reproducible.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (16512, 8)
Testing features: (4128, 8)
Training target: (16512,)
Testing target: (4128,)


**Expected output shapes**

- Training features: `(16512, 8)`
- Testing features: `(4128, 8)`
- Training target: `(16512,)`
- Testing target: `(4128,)`

The test set is not used during hyperparameter tuning. It is kept for an independent evaluation of the selected model.

## 9. Build the KNN regression pipeline

A `Pipeline` combines preprocessing and model training into one workflow.

1. The `preprocessor` step scales the input features.
2. The `model` step fits a `KNeighborsRegressor`.

Using a pipeline ensures the same preprocessing procedure is applied consistently during cross-validation, final fitting, and prediction.

In [9]:
knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsRegressor())
    ]
)

print("KNN regression pipeline created successfully.")

KNN regression pipeline created successfully.


## 10. Define the hyperparameter grid

We will test the exact values specified in the assignment:

- `n_neighbors`: number of neighbouring observations considered.
- `weights`: whether neighbours contribute equally (`uniform`) or closer neighbours receive more influence (`distance`).
- `p`: distance metric parameter. `p=1` corresponds to Manhattan distance; `p=2` corresponds to Euclidean distance.

There are 4 × 2 × 2 = **16 combinations**.

In [10]:
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

print("Hyperparameter grid:")
print(param_grid)

number_of_combinations = (
    len(param_grid["model__n_neighbors"])
    * len(param_grid["model__weights"])
    * len(param_grid["model__p"])
)

print("\nNumber of parameter combinations:", number_of_combinations)
print("Number of cross-validation fits:", number_of_combinations * 5)

Hyperparameter grid:
{'model__n_neighbors': [3, 5, 7, 9], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2]}

Number of parameter combinations: 16
Number of cross-validation fits: 80


## 11. Configure GridSearchCV

`GridSearchCV` evaluates every parameter combination using cross-validation.

- `cv=5` uses five folds.
- `scoring="r2"` selects the combination with the highest mean cross-validation R².
- `n_jobs=2` uses up to two parallel jobs to limit resource usage on a local computer.
- `verbose=2` prints progress messages so you can see the search advancing.

With 16 combinations and 5 folds, GridSearchCV performs **80 cross-validation fits**. It then refits the best configuration on the full training set.

In [11]:
grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=2,
    verbose=2
)

print("GridSearchCV configured successfully.")

GridSearchCV configured successfully.


## 12. Fit GridSearchCV

This is the most computationally intensive step. KNN repeatedly calculates distances between observations, so the search may take several minutes or longer depending on your computer.

Run this cell and allow it to finish. The progress messages from `verbose=2` will show the cross-validation fits.

The test data is not passed to `fit()`, so it remains separate from hyperparameter selection.

In [12]:
grid_search.fit(X_train, y_train)

print("\nHyperparameter tuning completed!")

Fitting 5 folds for each of 16 candidates, totalling 80 fits

Hyperparameter tuning completed!


## 13. Identify the best hyperparameters

`best_params_` returns the parameter combination with the highest mean cross-validation R².

`best_score_` returns that best mean cross-validation score.

The values are produced by your actual run; they are not fixed in advance.

In [13]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation R2 Score:")
print(grid_search.best_score_)

Best Parameters:
{'model__n_neighbors': 9, 'model__p': 1, 'model__weights': 'distance'}

Best Cross-Validation R2 Score:
0.7312668709861642


**Interpretation**

A higher cross-validation R² indicates that the selected configuration explained more variation in the target across the validation folds. This is a cross-validation result on the training portion, not the final test score.

## 14. Evaluate the final model on the test set

`best_estimator_` is the pipeline configured with the best hyperparameters and refitted on the complete training set.

We use it to predict the target values for `X_test`. Then we calculate the R² score by comparing predictions with the actual `y_test` values.

The test R² provides an evaluation on data that was held out from model fitting and hyperparameter selection.

In [14]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

test_r2 = r2_score(y_test, y_pred)

print("Test R2 Score:", test_r2)

Test R2 Score: 0.72210916268423


**How to interpret R²**

- R² = 1: predictions perfectly match the observed target values.
- R² = 0: the model performs no better, in squared-error terms, than predicting the test-set target mean.
- R² below 0: the model performs worse than that mean-baseline comparison.

A test score can differ from the best cross-validation score because it is calculated on a separate held-out dataset.

## 15. Compare actual and predicted values

We create a small table to inspect the first ten actual target values alongside the model's predictions.

Remember that both columns are in units of $100,000 because that is how the dataset's target is represented.

In [15]:
comparison = pd.DataFrame({
    "Actual Price (in $100,000s)": y_test.values,
    "Predicted Price (in $100,000s)": y_pred
})

comparison.head(10)

,"Actual Price (in $100,000s)","Predicted Price (in $100,000s)"
0,0.47700,0.509973
1,0.45800,0.937236
2,5.00001,4.421740
3,2.18600,2.704233
4,2.78000,2.704506
5,1.58700,2.116799
6,1.98200,2.573868
7,1.57500,1.853843
8,3.40000,2.174151
9,4.46600,4.898019


## 16. Save the trained pipeline

We use `joblib.dump()` to save the entire best pipeline, including the fitted preprocessing transformer and KNN model.

Saving the pipeline means the Streamlit application can load one file and pass it raw housing features. It does not need to recreate or separately fit the scaler.

In [16]:
joblib.dump(best_model, MODEL_PATH)

print("Model saved successfully!")
print("Saved model path:", MODEL_PATH)

Model saved successfully!
Saved model path: c:\Users\Admin\Documents\California-housing-price-prediction\california-housing-price-prediction\models\california_housing_model.pkl


## 17. Load and verify the saved model

We load the saved pipeline from disk and generate predictions on the same test features.

The loaded model's R² should match the earlier test R², apart from negligible floating-point differences. This verifies that the saved file can be loaded and used for prediction.

In [17]:
loaded_model = joblib.load(MODEL_PATH)

loaded_predictions = loaded_model.predict(X_test)
loaded_r2 = r2_score(y_test, loaded_predictions)

print("Loaded Model R2 Score:", loaded_r2)
print("Original Test R2 Score:", test_r2)
print("Scores match:", abs(loaded_r2 - test_r2) < 1e-10)

Loaded Model R2 Score: 0.72210916268423
Original Test R2 Score: 0.72210916268423
Scores match: True


## 18. Final summary

The following cell prints the main results from this run. Use these actual results when documenting the assignment and updating the GitHub README.

In [18]:
print("CALIFORNIA HOUSING MODEL SUMMARY")
print("--------------------------------")
print("Dataset rows:", len(df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", len(numerical_features))
print("Cross-validation folds:", 5)
print("Parameter combinations:", number_of_combinations)
print("Total CV fits:", number_of_combinations * 5)
print("Best parameters:", grid_search.best_params_)
print("Best CV R2:", round(grid_search.best_score_, 4))
print("Test R2:", round(test_r2, 4))
print("Model file:", MODEL_PATH)

CALIFORNIA HOUSING MODEL SUMMARY
--------------------------------
Dataset rows: 20640
Training rows: 16512
Testing rows: 4128
Features: 8
Cross-validation folds: 5
Parameter combinations: 16
Total CV fits: 80
Best parameters: {'model__n_neighbors': 9, 'model__p': 1, 'model__weights': 'distance'}
Best CV R2: 0.7313
Test R2: 0.7221
Model file: c:\Users\Admin\Documents\California-housing-price-prediction\california-housing-price-prediction\models\california_housing_model.pkl


## 19. Streamlit deployment

The Streamlit application is maintained in a separate `app.py` file in the project root.

After running this notebook and confirming that the model file exists, open the VS Code terminal at the project root and run:

```bash
python -m pip install streamlit
streamlit run app.py
```

The application loads `models/california_housing_model.pkl` and accepts the same eight input features used during training.

**Important:** Run the notebook from the project so that the `models` folder is created at the project root. The Streamlit app expects the saved model at that location.